# 02 - Hiperparámetros, Curva de Validación y Limitaciones de KNN en Regresión

## Curso de Machine Learning: Modelos de Regresión

En este cuaderno analizaremos el ajuste de hiperparámetros en `KNeighborsRegressor` mediante curvas de validación y `GridSearchCV`, y exploraremos dos limitaciones teóricas fundamentales del modelo:
1. **La Incapacidad de Extrapolación**: Por qué KNN no puede predecir tendencias fuera del rango numérico observado.
2. **La Maldición de la Dimensionalidad**: Cómo la dispersión en altas dimensiones degrada la noción de vecindad local.

---

### Contenidos
1. Dilema Sesgo-Varianza en Regresión Continua
2. Curva de Validación para el Hiperparámetro $K$
3. Búsqueda por Rejilla (`GridSearchCV`)
4. Demostración Práctica: Incapacidad de Extrapolación de KNN vs. Regresión Lineal
5. Conclusiones


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, validation_curve, GridSearchCV
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)


## 1. Curva de Validación para $K$ en Auto MPG

In [ ]:
df = pd.read_csv("../../data/auto_mpg.csv")
features = ['cylinders', 'displacement', 'horsepower', 'weight', 'acceleration', 'model_year']
X = df[features]
y = df['mpg']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

pipeline_knn = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsRegressor())
])

k_range = np.arange(1, 31)
train_scores, val_scores = validation_curve(
    pipeline_knn, X_train, y_train,
    param_name='knn__n_neighbors',
    param_range=k_range,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

train_m = np.mean(train_scores, axis=1)
val_m = np.mean(val_scores, axis=1)

plt.figure(figsize=(10, 4.8))
plt.plot(k_range, train_m, 'o-', color='#2b5c8f', lw=2, label='R² Entrenamiento')
plt.plot(k_range, val_m, 's-', color='#d95f02', lw=2, label='R² Validacion Cruzada (5-Fold)')

best_k = k_range[np.argmax(val_m)]
plt.axvline(best_k, color='green', linestyle='--', label=f'K Optimo = {best_k} (R² CV: {max(val_m):.4f})')
plt.xlabel('Numero de Vecinos (K)', fontweight='bold')
plt.ylabel('Coeficiente de Determinacion R²', fontweight='bold')
plt.title('Curva de Validacion: Compromiso Sesgo-Varianza en KNN Regresion', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()


## 2. Optimización Exhaustiva con `GridSearchCV`

Evaluamos simultáneamente el número de vecinos $K$, el esquema de ponderación y la métrica de distancia (Manhattan con $p=1$ vs. Euclidiana con $p=2$).


In [ ]:
param_grid = {
    'knn__n_neighbors': [3, 5, 7, 9, 11, 15, 20],
    'knn__weights': ['uniform', 'distance'],
    'knn__p': [1, 2]
}

grid_knn = GridSearchCV(pipeline_knn, param_grid, cv=5, scoring='r2', n_jobs=-1)
grid_knn.fit(X_train, y_train)

print(f"Mejores Hiperparametros: {grid_knn.best_params_}")
print(f"Mejor R2 CV:             {grid_knn.best_score_:.4f}")

best_model = grid_knn.best_estimator_
y_pred = best_model.predict(X_test)
print(f"R2 en Prueba:            {r2_score(y_test, y_pred):.4f}")
print(f"RMSE en Prueba:          {np.sqrt(mean_squared_error(y_test, y_pred)):.2f} mpg")


## 3. Limitación Fundamental: Incapacidad de Extrapolación

Una de las debilidades teóricas más importantes de los modelos basados en instancias (KNN y Árboles) es que **no pueden extrapolar tendencias fuera del rango numérico visto durante el entrenamiento**:
- Para cualquier consulta $\mathbf{x}^*$ situada muy lejos de los datos, la predicción es simplemente el promedio de los vecinos más externos.
- En contraste, un modelo paramétrico como la **Regresión Lineal** proyecta la pendiente indefinidamente.


In [ ]:
# Experimento de Extrapolacion: Serie creciente lineal
x_train_ext = np.linspace(1, 10, 40).reshape(-1, 1)
y_train_ext = 2.5 * x_train_ext.ravel() + np.random.normal(0, 1.0, 40)

# Datos futuros (zona de extrapolacion x in [10, 18])
x_future = np.linspace(10, 18, 50).reshape(-1, 1)
y_future_true = 2.5 * x_future.ravel()

# Modelos
lin_reg = LinearRegression().fit(x_train_ext, y_train_ext)
knn_reg = KNeighborsRegressor(n_neighbors=5, weights='distance').fit(x_train_ext, y_train_ext)

plt.figure(figsize=(11, 5))
plt.scatter(x_train_ext, y_train_ext, color='#2b5c8f', label='Datos Entrenamiento (x in [1, 10])', zorder=3)
plt.plot(x_future, y_future_true, 'k:', label='Tendencia Real Futura')
plt.plot(x_future, lin_reg.predict(x_future), color='#2ca02c', lw=2.5, label='Regresion Lineal (Extrapola)')
plt.plot(x_future, knn_reg.predict(x_future), color='#d62728', lw=2.5, label='KNN (Falla al Extrapolar: Meseta Plana)')

plt.axvline(10, color='gray', linestyle='--', alpha=0.7, label='Frontera de Datos de Entrenamiento')
plt.xlabel('Variable Predictora x', fontweight='bold')
plt.ylabel('Variable Objetivo y', fontweight='bold')
plt.title('Falla Estructural de KNN en Extrapolacion: No detecta tendencias crecientes externas', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()


## 4. Conclusiones

1. **Elección de K**: $K$ pequeño sobreajusta; $K$ grande sesga hacia la media global.
2. **Distancia**: La métrica Manhattan ($p=1$) suele ofrecer mejor rendimiento que la Euclidiana cuando existen múltiples variables correlacionadas.
3. **Cuidado con la Extrapolación**: Nunca uses KNN para proyecciones temporales futuras o escenarios donde las variables superen el dominio conocido de entrenamiento.
